# Part 1: Code Q&A System with Bash Tools

Answers questions about the `mcp-gateway-registry` codebase using bash tools for context retrieval.

**Architecture:** User Query → Query Router → Bash Tools → LLM Answer

All pipeline logic lives in `src/part1_pipeline.py`.

## Step 0: Clone the Repository

In [1]:
# Clone the mcp-gateway-registry codebase (run only once)
import os

if not os.path.exists("./mcp-gateway-registry"):
    os.system("git clone https://github.com/agentic-community/mcp-gateway-registry")
    print("Repository cloned successfully.")
else:
    print("Repository already exists, skipping clone.")


Repository already exists, skipping clone.


## Step 1: Setup

In [2]:
import sys
import os

# Make sure the project root is on the path so src/ can be imported
sys.path.insert(0, os.path.abspath(".."))

from src.part1_pipeline import answer_question, classify_query
from src.config import REPO_PATH, GROQ_MODEL

print("Imports OK")
print(f"  Repo  : {REPO_PATH}")
print(f"  Model : {GROQ_MODEL}")


Imports OK
  Repo  : ./mcp-gateway-registry
  Model : llama-3.3-70b-versatile


## Step 2: Verify the Router

Quick sanity check — confirm the router correctly classifies a simple question.

In [3]:
import json

# Test router on one question before running all 6
test = classify_query("What Python dependencies does this project use?")
print(json.dumps(test, indent=2))


{
  "query_type": "dependency",
  "reasoning": "Read pyproject.toml to find declared Python dependencies.",
  "commands": [
    "cat ./mcp-gateway-registry/pyproject.toml",
    "grep --include='*.py' -r 'import' ./mcp-gateway-registry | head -20"
  ]
}


## Step 3: Questions 1–3 (Simple)

Dependency lookup, entry point, and file structure — answered with `cat` and `tree`.

In [4]:
simple_questions = [
    "What Python dependencies does this project use?",
    "What is the main entry point file for the registry service?",
    "What programming languages and file types are used in this repository? (e.g., Python, TypeScript, YAML, JSON, Dockerfile, etc.)",
]

RESULTS_FILE = "./part1_results.txt"

# Start results file fresh
with open(RESULTS_FILE, "w") as f:
    f.write("Part 1: Code Q&A System Results\n" + "=" * 60 + "\n\n")

for i, question in enumerate(simple_questions, 1):
    answer = answer_question(question)
    print(f"\n{'='*60}\n[ANSWER]\n{'='*60}")
    print(answer)
    # Save immediately after each answer
    with open(RESULTS_FILE, "a") as f:
        f.write("Q" + str(i) + ": " + question + "\n" + "=" * 60 + "\n" + answer + "\n\n")
    print("[Saved Q" + str(i) + "]")


Question: What Python dependencies does this project use?

[Retrieving context...]
[Router] Query type : dependency
[Router] Reasoning  : Read pyproject.toml to find declared Python dependencies.
[Router] Executing 2 command(s):
  1. cat ./mcp-gateway-registry/pyproject.toml
  2. grep --include='*.txt' -r 'requires' ./mcp-gateway-registry || echo 'no requirements.txt'

[Generating answer...]

[ANSWER]
The project `mcp-registry` uses a variety of Python dependencies, which are specified in the `pyproject.toml` file. 

The required dependencies are listed under the `[project.dependencies]` section:
```toml
dependencies = [
    "aiofiles>=24.1.0",
    "fastapi>=0.115.12",
    "itsdangerous>=2.2.0",
    "jinja2>=3.1.6",
    "mcp>=1.9.3",
    "pydantic>=2.11.3",
    "pydantic-settings>=2.0.0",
    "httpx>=0.27.0",
    "python-dotenv>=1.1.0",
    "python-multipart>=0.0.20",
    "uvicorn[standard]>=0.34.2",
    "faiss-cpu>=1.7.4",
    "sentence-transformers>=3.0.0",
    "websockets>=15.0.1",

## Step 4: Questions 4–5 (Difficult)

Multi-file search using `grep` + `cat` combinations.

In [5]:
difficult_questions = [
    "How does the authentication flow work, from token validation to user authorization?",
    "What are all the API endpoints available in the registry service and what scopes do they require?",
]

for i, question in enumerate(difficult_questions, 4):
    answer = answer_question(question)
    print("\n" + "="*60 + "\n[ANSWER]\n" + "="*60)
    print(answer)
    with open("./part1_results.txt", "a") as f:
        f.write("Q" + str(i) + ": " + question + "\n" + "=" * 60 + "\n" + answer + "\n\n")
    print("[Saved Q" + str(i) + "]")


Question: How does the authentication flow work, from token validation to user authorization?

[Retrieving context...]
[Router] Query type : code_search
[Router] Reasoning  : Search for authentication-related keywords in Python files to understand the authentication flow.
[Router] Executing 5 command(s):
  1. grep -r 'token validation' ./mcp-gateway-registry --include='*.py'
  2. grep -r 'user authorization' ./mcp-gateway-registry --include='*.py'
  3. cat ./mcp-gateway-registry/registry/auth.py | grep -n 'validate_token'
  4. cat ./mcp-gateway-registry/registry/auth.py | grep -n 'authorize_user'
  5. grep -r 'OAuth' ./mcp-gateway-registry --include='*.py' --include='*.sh'

[Generating answer...]

[ANSWER]
Based on the provided codebase context, the authentication flow involves token validation, but the exact process from token validation to user authorization is not fully clear. Here's what can be gathered from the available information:

1. **Token Validation**: The codebase mention

## Step 5: Question 6 (Very Hard)

Synthesizes code, docs, and config across multiple directories.

In [6]:
hard_question = (
    "How would you add support for a new OAuth provider (e.g., Okta) to the authentication system? "
    "What files would need to be modified and what interfaces must be implemented?"
)

answer = answer_question(hard_question)
print("\n" + "="*60 + "\n[ANSWER]\n" + "="*60)
print(answer)
with open("./part1_results.txt", "a") as f:
    f.write("Q6: " + hard_question + "\n" + "=" * 60 + "\n" + answer + "\n\n")
print("[Saved Q6]")
print("\n✅ All results saved to part1_results.txt")


Question: How would you add support for a new OAuth provider (e.g., Okta) to the authentication system? What files would need to be modified and what interfaces must be implemented?
[Warning] Router JSON parse failed, using fallback.
Raw: 
{
    "query_type": "multi",
    "reasoning": "To add support for a new OAuth provider, we need to search for OAuth-related code, identify the interfaces that must be implemented, and find the files that need to be modified.",
    "commands": [
        "grep -r 'OAuth' ./mcp-gateway-registry --include='*.py' --include='*.sh' -l",
        "find ./mcp-gateway-registry/credentials-provider -name '*.py'",
        "cat ./mcp-gateway-registry/credentials-provider/oauth.py | grep -E 'class|def'",
        "grep -r 'interface|abstract' ./mcp-gateway-registry --include='*.py' --include='*.sh' -l",
        "find ./mcp-gateway-registry/docs -name "*.md" | xargs grep -l 'OAuth' | head -5"
    ]
}


[Retrieving context...]
[Router] Query type : multi
[Router] Rea

## (Optional) Interactive Mode

In [7]:
# Uncomment to ask your own questions
# while True:
#     q = input("Ask a question (or exit): ").strip()
#     if q.lower() in ("exit", "quit", ""):
#         break
#     print(answer_question(q))
